# 10 — Synthetic Population: PIDP × LA Counts

Builds a lightweight table of **how many times each PIDP appears in each Local Authority**
by mapping every synthetic individual (SIPHER) from their LSOA (`synthetic_zone`) up to LA.

**Inputs:**

- `data/1_pickle_sipher/sipher_optimized.pkl` — `synthetic_zone`, `pidp` (52 M rows)
- `data/0_raw/admin_geography_mappings.csv` — LSOA → LA lookup

**Output:**

- `data/10_synthetic_population/pidp_la_counts.csv` — `pidp`, `ladcd`, `ladnm`, `n`


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pathlib import Path
from data_pipeline.config_variables import DATA_FOLDER

# ── Paths ─────────────────────────────────────────────────────────────────────
SIPHER_PKL = Path(f"../{DATA_FOLDER}/1_pickle_sipher/sipher_optimized.pkl")
GEO_CSV    = Path(f"../{DATA_FOLDER}/0_raw/admin_geography_mappings.csv")
OUT_DIR    = Path(f"../{DATA_FOLDER}/10_synthetic_population")
OUT_CSV    = OUT_DIR / "pidp_la_counts.csv"

for p in [SIPHER_PKL, GEO_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"{p} — check pipeline prerequisites.")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load geography lookup ─────────────────────────────────────────────────────
print("Loading geography mappings …")
df_geo = (
    pd.read_csv(GEO_CSV, usecols=["lsoa21cd", "ladcd", "ladnm"], dtype=str, encoding="latin-1")
    .drop_duplicates(subset="lsoa21cd")
    .set_index("lsoa21cd")
)
print(f"  {len(df_geo):,} LSOAs → {df_geo['ladcd'].nunique():,} Local Authorities")

# ── Load sipher (2 cols only) and map to LA ───────────────────────────────────
print("\nLoading sipher pickle …")
df = pd.read_pickle(SIPHER_PKL)[["synthetic_zone", "pidp"]].copy()
df["pidp"] = pd.to_numeric(df["pidp"], errors="coerce").astype("int64")
print(f"  {len(df):,} rows, {df['pidp'].nunique():,} unique pidps")

print("Mapping LSOA → LA …")
df = df.join(df_geo, on="synthetic_zone", how="left")

n_no_geo = int(df["ladcd"].isna().sum())
if n_no_geo:
    pct = 100 * n_no_geo / len(df)
    print(f"  Warning: {n_no_geo:,} rows ({pct:.2f}%) could not be mapped to a geography")

# Drop unmapped rows and the LSOA column (no longer needed)
df = df.dropna(subset=["ladcd"]).drop(columns="synthetic_zone")

# ── Count pidp × LA ───────────────────────────────────────────────────────────
print("Counting pidp × LA …")
counts = (
    df.groupby(["pidp", "ladcd", "ladnm"], as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

print(f"\nResult: {len(counts):,} (pidp × LA) rows")
print(f"  unique pidps : {counts['pidp'].nunique():,}")
print(f"  unique LAs   : {counts['ladcd'].nunique():,}")
print(f"  n range      : {counts['n'].min():,} – {counts['n'].max():,}")
print(f"\nPreview (top 10 by n):")
print(counts.nlargest(10, 'n').to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────────
counts.to_csv(OUT_CSV, index=False)
print(f"\nSaved → {OUT_CSV}  ({OUT_CSV.stat().st_size / 1e6:.1f} MB)")

Loading geography mappings …
  43,501 LSOAs → 363 Local Authorities

Loading sipher pickle …
  52,853,971 rows, 27,330 unique pidps
Mapping LSOA → LA …
Counting pidp × LA …

Result: 7,969,122 (pidp × LA) rows
  unique pidps : 27,330
  unique LAs   : 346
  n range      : 1 – 1,432

Preview (top 10 by n):
      pidp     ladcd             ladnm    n
 614530975 E08000025        Birmingham 1432
 681143779 S12000036 City of Edinburgh 1411
1576165370 E08000025        Birmingham 1375
1429976767 E08000025        Birmingham 1206
 953730619 E09000025            Newham 1126
 953730615 E09000025            Newham 1079
 613001647 S12000049      Glasgow City 1074
1089918370 S12000049      Glasgow City 1062
 409895175 E08000025        Birmingham 1060
 480637690 S12000049      Glasgow City 1053

Saved → ../data/10_synthetic_population/pidp_la_counts.csv  (276.2 MB)
